**"이 제품 설명 써줘"라는 모호한 Baseline 프롬프트를 기초 방법론으로 개선하고, 개선된 프롬프트의 JSON 출력을 검증하라.**

* 테스트 상품 3개: 텀블러, 러닝화, 탁상용 가습기 (상품명 | 스펙 | 가격 형식의 한 줄 정보)

```
    products = [
        "텀블러 | 스테인리스 이중 진공, 500ml, 보온 12시간 | 29,000원",
        "러닝화 | 무게 220g, 쿠셔닝 폼, 통기성 메쉬 | 119,000원",
        "탁상용 가습기 | 초음파 방식, 300ml, 무드등, USB 전원 | 25,000원",
    ]
```

* 개선 프롬프트에 적용할 방법론

    * 페르소나 부여 (System)

    * 구분 기호 (<product> 태그)

    * 해석 여지 최소화 (글자 수 · 개수 제한)

    * 대체 경로 (정보에 없는 내용은 만들지 않기)

* 출력 형식: {"title", "summary", "features"(정확히 3개), "target"}

* 응답을 json.loads로 파싱하고, 필수 키 · features 개수 · title 길이(20자 이내)를 검사하는 validate() 함수를 작성한다.

* Baseline과 개선 프롬프트의 출력 토큰, 형식 준수율을 표로 비교한다.

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

import os
MY_API_KEY=os.environ['OPENAI_API_KEY']

client=OpenAI()
MODEL='gpt-6-luna'

In [10]:
products = [
        "텀블러 | 스테인리스 이중 진공, 500ml, 보온 12시간 | 29,000원",
        "러닝화 | 무게 220g, 쿠셔닝 폼, 통기성 메쉬 | 119,000원",
        "탁상용 가습기 | 초음파 방식, 300ml, 무드등, USB 전원 | 25,000원",
]

BASELINE = '이 제품 설명 써 줘: {p}'

In [11]:
SYSTEM = ('당신은 10년 차 이커머스 상품 상세페이지 카피라이터입니다. '
          '과장 없이 사실에 근거해 작성하고, 주어진 상품 정보에 없는 수치·기능·효과는 절대 만들어내지 않습니다. '
          '응답은 JSON 객체 하나만 출력합니다.')
USER = '''<product> 태그 안의 정보를 바탕으로 제품 상세페이지의 문구를 작성할 것
- title: 20자 이내
- summary: 1줄
- features: 핵심 특징 정확히 3개
- target: 추천 대상 1줄
아래 JSON 형식으로 답할 것. 주어진 정보 이외의 것을 사용하지 말 것.
{{'title': '', 'summary': '', 'features':  ['','',''], 'target': ''}}

<product>{p}</product>'''

REQUIRED = {'title','summary','features','target'}

In [12]:
import json
import re

def parse_json(text):
    # ```json 코드블록으로 감싸서 오는 경우 펜스 제거 후 json.loads
    cleaned = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    return json.loads(cleaned)

def validate(text):
    """응답 문자열을 파싱하고 (data, 오류 목록) 반환, 오류 목록이 비어 있으면 형식 준수"""
    try:
        data = parse_json(text)
    except json.JSONDecodeError:
        return None, ['JSON 파싱 실패']
    if not isinstance(data, dict):
        return None, ['JSON 객체 아님']

    errors = []
    if set(data) != REQUIRED:
        errors.append(f'키 불일치: {sorted(data)}')
    features = data.get('features')
    if not isinstance(features, list) or len(features) != 3:
        errors.append(f'features 개수: {len(features) if isinstance(features, list) else "리스트 아님"}')
    title = data.get('title')
    if not isinstance(title, str) or len(title) > 20:
        errors.append(f'title 길이: {len(str(title))}자')
    return data, errors

In [13]:
rows = []
outputs = {}  # (프롬프트 종류, 상품명) -> 원문 응답

for p in products:
    name = p.split(' | ')[0]
    base = client.responses.create(model=MODEL, input=BASELINE.format(p=p))
    imp = client.responses.create(model=MODEL, instructions=SYSTEM, input=USER.format(p=p))

    # Baseline도 같은 validate()로 검사해 형식 준수율 비교
    for kind, res in [('Baseline', base), ('Improved', imp)]:
        data, errors = validate(res.output_text)
        outputs[(kind, name)] = res.output_text
        rows.append({
            'prompt': kind,
            'product': name,
            'output_tokens': res.usage.output_tokens,
            'parsed': data is not None,
            'valid': not errors,
            'errors': ', '.join(errors),
            'title': data.get('title') if data else None,
        })

In [14]:
# 개선 프롬프트 응답 원문 확인
for p in products:
    name = p.split(' | ')[0]
    print(f'[{name}]')
    print(outputs[('Improved', name)])
    print()

[텀블러]
{"title":"스테인리스 진공 텀블러","summary":"500ml 용량에 보온 12시간을 갖춘 텀블러입니다.","features":["스테인리스 이중 진공 구조","500ml 용량","보온 12시간"],"target":"보온 기능과 500ml 용량의 텀블러를 찾는 분께 추천합니다."}

[러닝화]
{"title":"220g 러닝화","summary":"쿠셔닝 폼과 통기성 메쉬를 적용한 러닝화입니다.","features":["무게 220g","쿠셔닝 폼","통기성 메쉬"],"target":"러닝화를 찾는 분"}

[탁상용 가습기]
{"title":"무드등 탁상용 가습기","summary":"초음파 방식의 300ml 탁상용 가습기입니다.","features":["초음파 방식","300ml 용량","무드등 및 USB 전원"],"target":"USB 전원으로 사용할 탁상용 가습기를 찾는 분께 추천합니다."}



In [15]:
# Baseline 응답 원문 확인 (앞부분만)
for p in products:
    name = p.split(' | ')[0]
    print(f'[{name}]')
    print(outputs[('Baseline', name)][:300], '...')
    print()

[텀블러]
**하루 종일 든든한 온기, 스테인리스 진공 텀블러**

스테인리스 이중 진공 구조로 음료의 따뜻함을 최대 12시간 유지해 줍니다. 넉넉한 500ml 용량으로 커피부터 차까지 여유롭게 담아 외출이나 출근길에 함께하기 좋아요.

- 용량: 500ml  
- 소재·구조: 스테인리스 이중 진공  
- 보온: 최대 12시간  
- 가격: 29,000원 ...

[러닝화]
### 가볍게 달리고, 편안하게

220g의 가벼운 무게로 발걸음은 산뜻하게, 쿠셔닝 폼으로 착지는 부드럽게. 통기성 메쉬 소재가 쾌적한 착용감을 선사하는 러닝화입니다.

- **무게:** 220g  
- **특징:** 쿠셔닝 폼, 통기성 메쉬  
- **가격:** 119,000원 ...

[탁상용 가습기]
### 촉촉한 분위기를 더하는 탁상용 초음파 가습기

책상이나 침대 옆에 두고 사용하기 좋은 컴팩트한 가습기입니다. 초음파 방식으로 분무하며, 은은한 무드등이 편안한 분위기를 연출해 줍니다. USB 전원으로 간편하게 사용할 수 있어 사무실이나 집 어디에나 잘 어울립니다.

- **방식:** 초음파식  
- **용량:** 300ml  
- **기능:** 무드등  
- **전원:** USB  
- **가격:** 25,000원 ...



In [16]:
import pandas as pd

df = pd.DataFrame(rows)
df

,prompt,product,output_tokens,parsed,valid,errors,title
0,Baseline,텀블러,158,False,False,JSON 파싱 실패,NaN
1,Improved,텀블러,260,True,True,,스테인리스 진공 텀블러
2,Baseline,러닝화,139,False,False,JSON 파싱 실패,NaN
3,Improved,러닝화,136,True,True,,220g 러닝화
4,Baseline,탁상용 가습기,213,False,False,JSON 파싱 실패,NaN
5,Improved,탁상용 가습기,258,True,True,,무드등 탁상용 가습기


In [17]:
# Baseline vs 개선 프롬프트 비교표
summary = df.groupby('prompt').agg(
    평균_출력토큰=('output_tokens', 'mean'),
    총_출력토큰=('output_tokens', 'sum'),
    파싱_성공률=('parsed', 'mean'),
    형식_준수율=('valid', 'mean'),
)
summary['평균_출력토큰'] = summary['평균_출력토큰'].round(1)
summary['파싱_성공률'] = summary['파싱_성공률'].map('{:.0%}'.format)
summary['형식_준수율'] = summary['형식_준수율'].map('{:.0%}'.format)
summary

,평균_출력토큰,총_출력토큰,파싱_성공률,형식_준수율
prompt,,,,
Baseline,170.0,510,0%,0%
Improved,218.0,654,100%,100%


---

**쇼핑몰 정책 문서(배송, 교환/반품, 환불, 적립금)만 근거로 고객 질문에 답하는 상담 프롬프트를 만들고, 정책에 없는 질문에 답을 지어내지 않는지 테스트하라.**

### POLICY
- [배송] 평일 오후 2시 이전 주문은 당일 출고, 이후 주문은 다음 영업일 출고. 제주 · 도서산간은 3,000원 추가.
- [교환/반품] 수령 후 7일 이내 가능. 단순 변심은 왕복 배송비 6,000원 고객 부담, 제품 불량은 무료.
- [환불] 반품 확인 후 카드 결제는 3~5영업일, 계좌이체는 1~2영업일 내 환불.
- [적립금] 구매 확정 시 결제 금액의 1% 적립, 유효기간 1년.

    * System: 상담원 페르소나, "Context에만 근거", 근거가 없으면 "정책에서 확인할 수 없습니다" + 상담원 연결

    * User: 강의의 USER 템플릿 구조 (지시사항 + 출력 명세 / # Context / # Input Data)

    * 출력 형식: {"answer": "", "evidence": "근거 정책 항목명 또는 null", "need_human": true/false}

    * 테스트 질문 6개: 정책으로 답할 수 있는 질문 3개 + 정책에 없는 질문 3개 (예: 해외 배송, 적립금 현금화, 회원 등급 할인)

tests = [  # (질문, 정책으로 답변 가능 여부)
    ("오후 3시에 주문하면 언제 출고되나요?", True),
    ("단순 변심으로 반품하면 배송비는 얼마예요?", True),
    ("계좌이체로 결제했는데 환불 얼마나 걸려요?", True),
    ("해외 배송도 되나요?", False),
    ("적립금을 현금으로 바꿀 수 있나요?", False),
    ("회원 등급별 할인율이 궁금해요", False),
]
* 정책으로 답할 수 있는 질문은 need_human=false, 없는 질문은 true가 나와야 정답으로 보고 라우팅 정확도를 계산한다.

* 정책에 없는 질문에 답을 지어낸 건수를 따로 집계한다.

In [18]:
import json
import re

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-6-luna'

def parse_json(text):
    # ```json 코드블록으로 감싸서 오는 경우 펜스 제거 후 json.loads
    cleaned = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    return json.loads(cleaned)

In [19]:
POLICY = """[배송] 평일 오후 2시 이전 주문은 당일 출고, 이후 주문은 다음 영업일 출고. 제주 · 도서산간은 3,000원 추가.
[교환/반품] 수령 후 7일 이내 가능. 단순 변심은 왕복 배송비 6,000원 고객 부담, 제품 불량은 무료.
[환불] 반품 확인 후 카드 결제는 3~5영업일, 계좌이체는 1~2영업일 내 환불.
[적립금] 구매 확정 시 결제 금액의 1% 적립, 유효기간 1년."""

tests = [  # (질문, 정책으로 답변 가능 여부)
    ("오후 3시에 주문하면 언제 출고되나요?", True),
    ("단순 변심으로 반품하면 배송비는 얼마예요?", True),
    ("계좌이체로 결제했는데 환불 얼마나 걸려요?", True),
    ("해외 배송도 되나요?", False),
    ("적립금을 현금으로 바꿀 수 있나요?", False),
    ("회원 등급별 할인율이 궁금해요", False),
]

In [20]:
FALLBACK = '정책에서 확인할 수 없습니다'

# 상담원 페르소나 + Context 근거 제한 + 대체 경로
QA_SYSTEM = f"""당신은 온라인 쇼핑몰 고객센터 상담원입니다.
- 반드시 # Context에 있는 정책 내용만 근거로 답변합니다.
- 정책에서 근거를 찾을 수 없으면 추측하지 않고, answer에 "{FALLBACK}. 상담원에게 연결해 드리겠습니다."라고 쓰고 evidence는 null, need_human은 true로 합니다.
- 정책에 근거가 있으면 need_human은 false입니다.
- 정중한 존댓말로, answer는 2문장 이내로 작성합니다."""

# 지시사항 + 출력 명세 / # Context / # Input Data
QA_USER = """# Context의 쇼핑몰 정책을 참고하여 # Input Data의 고객 질문에 답할 것.
- evidence: 근거가 된 정책 항목명 (예: "배송"), 근거가 없으면 null
- need_human: 정책으로 답할 수 없으면 true, 답할 수 있으면 false
반드시 아래 JSON 형식으로만 답할 것. 코드블록이나 다른 설명은 쓰지 말 것.
{{"answer": "...", "evidence": "정책 항목명" 또는 null, "need_human": true 또는 false}}

# Context
{context}

# Input Data
{question}"""

QA_REQUIRED = {'answer', 'evidence', 'need_human'}

In [21]:
def validate_qa(text):
    """상담 응답을 파싱하고 (data, 오류 목록) 반환"""
    try:
        data = parse_json(text)
    except json.JSONDecodeError:
        return None, ['JSON 파싱 실패']
    if not isinstance(data, dict):
        return None, ['JSON 객체 아님']

    errors = []
    if set(data) != QA_REQUIRED:
        errors.append(f'키 불일치: {sorted(data)}')
    if not isinstance(data.get('need_human'), bool):
        errors.append(f'need_human 타입: {type(data.get("need_human")).__name__}')
    if data.get('evidence') is not None and not isinstance(data.get('evidence'), str):
        errors.append('evidence가 문자열/null 아님')
    return data, errors

In [22]:
qa_rows = []

for q, answerable in tests:
    res = client.responses.create(
        model=MODEL, instructions=QA_SYSTEM,
        input=QA_USER.format(context=POLICY, question=q),
    )
    data, errors = validate_qa(res.output_text)
    need_human = data.get('need_human') if data else None
    answer = str(data.get('answer', '')) if data else res.output_text

    qa_rows.append({
        'question': q,
        'answerable': answerable,
        'format_ok': not errors,
        'need_human': need_human,
        'evidence': data.get('evidence') if data else None,
        # 답변 가능 질문은 need_human=False, 정책에 없는 질문은 True여야 정답
        'routing_ok': need_human is (not answerable),
        # 정책에 없는 질문인데 대체 문구 없이 답을 만들어낸 경우
        'hallucinated': (not answerable) and (FALLBACK not in answer),
        'answer': answer,
        'errors': ', '.join(errors),
    })

qa_df = pd.DataFrame(qa_rows)
qa_df

,question,answerable,format_ok,need_human,evidence,routing_ok,hallucinated,answer,errors
0,오후 3시에 주문하면 언제 출고되나요?,True,True,False,배송,True,False,오후 3시 주문은 다음 영업일에 출고됩니다.,
1,단순 변심으로 반품하면 배송비는 얼마예요?,True,True,False,교환/반품,True,False,"단순 변심으로 반품하시면 왕복 배송비 6,000원을 부담하셔야 합니다.",
2,계좌이체로 결제했는데 환불 얼마나 걸려요?,True,True,False,환불,True,False,계좌이체 환불은 반품 확인 후 1~2영업일 내 처리됩니다.,
3,해외 배송도 되나요?,False,True,True,NaN,True,False,정책에서 확인할 수 없습니다. 상담원에게 연결해 드리겠습니다.,
4,적립금을 현금으로 바꿀 수 있나요?,False,True,True,NaN,True,False,정책에서 확인할 수 없습니다. 상담원에게 연결해 드리겠습니다.,
5,회원 등급별 할인율이 궁금해요,False,True,True,NaN,True,False,정책에서 확인할 수 없습니다. 상담원에게 연결해 드리겠습니다.,


In [23]:
# 답변 원문 확인
for r in qa_rows:
    print(f"Q. {r['question']}  (answerable={r['answerable']}, need_human={r['need_human']}, evidence={r['evidence']})")
    print(f"A. {r['answer']}")
    print()

Q. 오후 3시에 주문하면 언제 출고되나요?  (answerable=True, need_human=False, evidence=배송)
A. 오후 3시 주문은 다음 영업일에 출고됩니다.

Q. 단순 변심으로 반품하면 배송비는 얼마예요?  (answerable=True, need_human=False, evidence=교환/반품)
A. 단순 변심으로 반품하시면 왕복 배송비 6,000원을 부담하셔야 합니다.

Q. 계좌이체로 결제했는데 환불 얼마나 걸려요?  (answerable=True, need_human=False, evidence=환불)
A. 계좌이체 환불은 반품 확인 후 1~2영업일 내 처리됩니다.

Q. 해외 배송도 되나요?  (answerable=False, need_human=True, evidence=None)
A. 정책에서 확인할 수 없습니다. 상담원에게 연결해 드리겠습니다.

Q. 적립금을 현금으로 바꿀 수 있나요?  (answerable=False, need_human=True, evidence=None)
A. 정책에서 확인할 수 없습니다. 상담원에게 연결해 드리겠습니다.

Q. 회원 등급별 할인율이 궁금해요  (answerable=False, need_human=True, evidence=None)
A. 정책에서 확인할 수 없습니다. 상담원에게 연결해 드리겠습니다.



In [24]:
# 답변 가능 여부별 결과표
qa_summary = qa_df.groupby('answerable').agg(
    질문수=('question', 'count'),
    형식_준수율=('format_ok', 'mean'),
    라우팅_정확도=('routing_ok', 'mean'),
    지어낸_답변=('hallucinated', 'sum'),
)
qa_summary['형식_준수율'] = qa_summary['형식_준수율'].map('{:.0%}'.format)
qa_summary['라우팅_정확도'] = qa_summary['라우팅_정확도'].map('{:.0%}'.format)

print(f"전체 라우팅 정확도: {qa_df['routing_ok'].mean():.0%}")
print(f"정책에 없는 질문에 답을 지어낸 건수: {int(qa_df['hallucinated'].sum())} / {int((~qa_df['answerable']).sum())}")
qa_summary

전체 라우팅 정확도: 100%
정책에 없는 질문에 답을 지어낸 건수: 0 / 3


,질문수,형식_준수율,라우팅_정확도,지어낸_답변
answerable,,,,
False,3,100%,100%,0
True,3,100%,100%,0


---

**고객 문의 메시지에서 분류(category), 긴급도(urgency), 요약(summary)을 JSON으로 추출하고, Zero-shot과 Few-shot의 형식 · 판단 일관성을 비교하라.**

* category: 배송 / 결제 / 상품 / 계정 중 하나

* urgency: low / medium / high 중 하나

* 테스트 메시지 6개와 정답 레이블(category, urgency)을 직접 만든다.

* Zero-shot: 필드와 허용값만 설명한다.

* Few-shot: 긴급도 판단 기준 한 줄과, 입력 → JSON 출력 예시 3개를 추가한다.

* 평가 지표

    * format_ok: 파싱 성공 + 키 3개 정확 + 허용값 준수

    * category_ok, urgency_ok: 레이블 정확도

    * 평균 입력 토큰: 예시 추가 비용

두 프롬프트의 지표 평균을 표로 비교한다.

In [25]:
import json
import re

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-6-luna'

def parse_json(text):
    # ```json 코드블록으로 감싸서 오는 경우 펜스 제거 후 json.loads
    cleaned = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    return json.loads(cleaned)

In [26]:
CATEGORIES = ['배송', '결제', '상품', '계정']
URGENCIES = ['low', 'medium', 'high']
EX_REQUIRED = {'category', 'urgency', 'summary'}

messages = [  # (메시지, 정답 category, 정답 urgency)
    ("어제 결제했는데 카드 승인이 두 번 났어요. 한 건 빨리 취소해 주세요", "결제", "high"),
    ("로그인하려는데 인증 메일이 안 와요. 다시 보내 주실 수 있나요?", "계정", "medium"),
    ("주문한 지 일주일째인데 배송 조회가 안 돼요. 이번 주 토요일 행사에 써야 해요", "배송", "high"),
    ("이 원피스 세탁기에 돌려도 되나요?", "상품", "low"),
    ("받는 주소를 회사로 바꾸고 싶어요", "배송", "medium"),
    ("무통장 입금도 되나요?", "결제", "low"),
]

In [27]:
FIELD_SPEC = f"""고객 메시지를 분석해서 JSON으로만 답할 것. 코드블록이나 다른 설명은 쓰지 말 것.
- category: {' / '.join(CATEGORIES)} 중 하나
- urgency: {' / '.join(URGENCIES)} 중 하나
- summary: 문의 내용 한 줄 요약
형식: {{{{"category": "", "urgency": "", "summary": ""}}}}"""

# Zero-shot: 필드와 허용값만 설명
ZERO_SHOT = FIELD_SPEC + """

메시지: {m}"""

# Few-shot: 긴급도 판단 기준 한 줄 + 입력 → 출력 예시 3개
FEW_SHOT = FIELD_SPEC + """
urgency 기준: 금전 피해 · 일정 임박 = high, 처리 · 변경 요청 = medium, 단순 문의 = low

메시지: 니트 색상이 사진이랑 너무 달라요. 교환 접수해 주세요
{{"category": "상품", "urgency": "medium", "summary": "니트 색상 차이로 교환 요청"}}

메시지: 닉네임은 어디서 바꾸나요?
{{"category": "계정", "urgency": "low", "summary": "닉네임 변경 방법 문의"}}

메시지: 결제는 됐는데 주문 내역이 안 보여요. 오늘 마감 한정판이라 걱정돼요
{{"category": "결제", "urgency": "high", "summary": "결제 후 주문 내역 누락, 당일 마감 상품"}}

메시지: {m}"""

print(FEW_SHOT.format(m=messages[0][0]))

고객 메시지를 분석해서 JSON으로만 답할 것. 코드블록이나 다른 설명은 쓰지 말 것.
- category: 배송 / 결제 / 상품 / 계정 중 하나
- urgency: low / medium / high 중 하나
- summary: 문의 내용 한 줄 요약
형식: {"category": "", "urgency": "", "summary": ""}
urgency 기준: 금전 피해 · 일정 임박 = high, 처리 · 변경 요청 = medium, 단순 문의 = low

메시지: 니트 색상이 사진이랑 너무 달라요. 교환 접수해 주세요
{"category": "상품", "urgency": "medium", "summary": "니트 색상 차이로 교환 요청"}

메시지: 닉네임은 어디서 바꾸나요?
{"category": "계정", "urgency": "low", "summary": "닉네임 변경 방법 문의"}

메시지: 결제는 됐는데 주문 내역이 안 보여요. 오늘 마감 한정판이라 걱정돼요
{"category": "결제", "urgency": "high", "summary": "결제 후 주문 내역 누락, 당일 마감 상품"}

메시지: 어제 결제했는데 카드 승인이 두 번 났어요. 한 건 빨리 취소해 주세요


In [29]:
def validate_extract(text):
    """추출 응답을 파싱하고 (data, 오류 목록) 반환, 오류 목록이 비어 있으면 format_ok"""
    try:
        data = parse_json(text)
    except json.JSONDecodeError:
        return None, ['JSON 파싱 실패']
    if not isinstance(data, dict):
        return None, ['JSON 객체 아님']

    errors = []
    if set(data) != EX_REQUIRED:
        errors.append(f'키 불일치: {sorted(data)}')
    if data.get('category') not in CATEGORIES:
        errors.append(f'category 허용값 아님: {data.get("category")}')
    if data.get('urgency') not in URGENCIES:
        errors.append(f'urgency 허용값 아님: {data.get("urgency")}')
    return data, errors

In [30]:
ex_rows = []

for prompt_name, tpl in [('Zero-shot', ZERO_SHOT), ('Few-shot', FEW_SHOT)]:
    for m, cat, urg in messages:
        res = client.responses.create(model=MODEL, input=tpl.format(m=m))
        data, errors = validate_extract(res.output_text)
        data = data or {}
        ex_rows.append({
            'prompt': prompt_name,
            'message': m,
            'format_ok': not errors,
            'category_ok': data.get('category') == cat,
            'urgency_ok': data.get('urgency') == urg,
            'label': f'{cat}/{urg}',
            'pred': f"{data.get('category')}/{data.get('urgency')}",
            'summary': data.get('summary'),
            'in_tok': res.usage.input_tokens,
            'errors': ', '.join(errors),
        })

ex_df = pd.DataFrame(ex_rows)
ex_df

,prompt,message,format_ok,category_ok,urgency_ok,label,pred,summary,in_tok,errors
0,Zero-shot,어제 결제했는데 카드 승인이 두 번 났어요. 한 건 빨리 취소해 주세요,True,True,True,결제/high,결제/high,어제 결제 건에서 카드 승인이 중복되어 한 건의 취소를 요청함.,111,
1,Zero-shot,로그인하려는데 인증 메일이 안 와요. 다시 보내 주실 수 있나요?,True,True,True,계정/medium,계정/medium,로그인 인증 메일이 도착하지 않아 재발송을 요청함,107,
2,Zero-shot,주문한 지 일주일째인데 배송 조회가 안 돼요. 이번 주 토요일 행사에 써야 해요,True,True,True,배송/high,배송/high,"주문한 지 일주일째 배송 조회가 되지 않으며, 이번 주 토요일 행사 전에 받아야 합니다.",115,
3,Zero-shot,이 원피스 세탁기에 돌려도 되나요?,True,True,True,상품/low,상품/low,원피스를 세탁기에 세탁해도 되는지 문의함,101,
4,Zero-shot,받는 주소를 회사로 바꾸고 싶어요,True,True,True,배송/medium,배송/medium,받는 주소를 회사 주소로 변경하고 싶어 합니다.,99,
5,Zero-shot,무통장 입금도 되나요?,True,True,True,결제/low,결제/low,무통장 입금이 가능한지 문의함,97,
6,Few-shot,어제 결제했는데 카드 승인이 두 번 났어요. 한 건 빨리 취소해 주세요,True,True,True,결제/high,결제/high,카드 중복 승인 건의 빠른 취소 요청,292,
7,Few-shot,로그인하려는데 인증 메일이 안 와요. 다시 보내 주실 수 있나요?,True,True,True,계정/medium,계정/medium,로그인 인증 메일 재발송 요청,288,
8,Few-shot,주문한 지 일주일째인데 배송 조회가 안 돼요. 이번 주 토요일 행사에 써야 해요,True,True,True,배송/high,배송/high,주문 후 일주일째 배송 조회가 되지 않으며 토요일 행사에 필요함,296,
9,Few-shot,이 원피스 세탁기에 돌려도 되나요?,True,True,True,상품/low,상품/low,원피스 세탁기 사용 가능 여부 문의,282,


In [31]:
# Zero-shot vs Few-shot 지표 평균 비교표
ex_summary = ex_df.groupby('prompt', sort=False).agg(
    format_ok=('format_ok', 'mean'),
    category_ok=('category_ok', 'mean'),
    urgency_ok=('urgency_ok', 'mean'),
    평균_입력토큰=('in_tok', 'mean'),
)
for col in ['format_ok', 'category_ok', 'urgency_ok']:
    ex_summary[col] = ex_summary[col].map('{:.0%}'.format)
ex_summary['평균_입력토큰'] = ex_summary['평균_입력토큰'].round(1)
ex_summary

,format_ok,category_ok,urgency_ok,평균_입력토큰
prompt,,,,
Zero-shot,100%,100%,100%,105.0
Few-shot,100%,100%,100%,286.0
